# AutoTrainer And Single-Node Multi-GPU Training

<a href="https://colab.research.google.com/github/openlanguagemodel/openlanguagemodel/blob/main/notebooks/04_autotrainer_distributed_colab.ipynb" target="_blank">
  <img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open in Colab"/>
</a>

This notebook shows how `AutoTrainer` chooses CPU, single-GPU, DDP,
or FSDP paths. It runs safely on a normal Colab runtime, while also
showing the exact single-node multi-GPU launch pattern.

Multi-node training is a v4 roadmap item; this notebook is about v2
single-machine scaling.

## Install OLM

In [ ]:
import importlib.util
import subprocess
import sys

if importlib.util.find_spec("olm") is None:
    subprocess.check_call([
        sys.executable,
        "-m",
        "pip",
        "install",
        "-q",
        "git+https://github.com/openlanguagemodel/openlanguagemodel.git",
    ])

## Imports And A Tiny Training Setup

In [ ]:
import random
from pathlib import Path

import torch

from olm.data.datasets import DataLoader, LocalTextDataset
from olm.data.tokenization import HFTokenizer
from olm.nn.blocks import LM
from olm.train import AutoTrainer
from olm.train.device import (
    TrainerStrategy,
    determine_strategy,
    detect_devices,
    parse_device_string,
    print_strategy_summary,
)
from olm.train.optim import AdamW

seed = 123
random.seed(seed)
torch.manual_seed(seed)

context_length = 128
device = "cuda" if torch.cuda.is_available() else "cpu"
print("torch cuda available:", torch.cuda.is_available())
print("torch cuda device count:", torch.cuda.device_count())

data_dir = Path("autotrainer_data")
data_dir.mkdir(exist_ok=True)
(data_dir / "train.txt").write_text(
    "\n".join(
        "AutoTrainer keeps the model readable while choosing the training path from the hardware."
        for _ in range(500)
    ),
    encoding="utf-8",
)

tokenizer = HFTokenizer("gpt2")
dataset = LocalTextDataset(data_dir, tokenizer, context_length=context_length, shuffle=True, seed=seed)
loader = DataLoader(dataset, batch_size=4, num_workers=0, pin_memory=device.startswith("cuda"))

model = LM(
    tokenizer.vocab_size,
    embed_dim=128,
    num_heads=4,
    num_layers=2,
    max_seq_len=context_length,
    dropout=0.0,
)

## Inspect Hardware And Strategy

In [ ]:
hardware = detect_devices()
print(hardware)

config = parse_device_string("auto", model=model)
config = determine_strategy(config, model=model, preset="balanced")
print_strategy_summary(config)

## Let AutoTrainer Configure The Run

In [ ]:
trainer = AutoTrainer(
    model,
    AdamW,
    loader,
    device="auto",
    context_length=context_length,
    learning_rate=3e-4,
    weight_decay=0.1,
    grad_accum_steps=2,
    use_amp=device.startswith("cuda"),
    preset="balanced",
    verbose=True,
)

print("selected trainer:", type(trainer).__name__)
losses = trainer.train(epochs=1, max_steps=3, log_interval=1)
print("losses:", losses)

## Single-Node Multi-GPU Pattern

In a notebook you normally have one Python process, so this section is
explanatory. For multiple GPUs on one machine, put the training code
in a script and launch it with `torchrun`.

In [ ]:
print("Single machine, 4 GPUs:")
print("torchrun --nproc_per_node=4 train.py")
print()
print("Inside train.py, keep the same AutoTrainer call:")
print('trainer = AutoTrainer(model, AdamW, loader, device="auto", context_length=1024)')

## Forcing A Strategy

In [ ]:
# Do not run this block unless the notebook was launched with torchrun
# on a machine that has multiple GPUs.
RUN_FORCED_MULTI_GPU_EXAMPLE = False

if RUN_FORCED_MULTI_GPU_EXAMPLE:
    trainer = AutoTrainer(
        model,
        AdamW,
        loader,
        device="auto",
        context_length=context_length,
        force_strategy=TrainerStrategy.MULTI_GPU_DDP,
        verbose=True,
    )
    trainer.train(epochs=1, max_steps=3)
else:
    print("Skipped forced DDP example. Use torchrun on a multi-GPU machine to run it.")

## Mental Model

- CPU or one GPU: `AutoTrainer` returns the base `Trainer`.
- Multiple GPUs on one machine: it can select `DDPTrainer` or
  `FSDPTrainer`.
- DDP is usually simpler and faster when the model fits on each GPU.
- FSDP helps when model states need to be sharded across GPUs.
- Multi-node launch/configuration is intentionally reserved for v4.